# Lesson 04 Lab — Continuous Batching, Throughput, and Fairness

**Puzzle:** When should a newly arrived short request enter a GPU batch?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Static batching waits for every sequence in a group to finish. Autoregressive sequences rarely finish together, so completed rows become empty work while new arrivals remain queued. Iteration-level scheduling can refill those slots, but an unconstrained throughput policy can starve old or large requests.


## 0. Predict before running

1. Predict which scheduler minimizes makespan.
2. Identify which policy hurts the oldest long request.
3. Choose an observable starvation gate.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

A discrete-event scheduler replays the same arrivals and token demands under static groups, shortest-remaining-token priority, and age-aware continuous batching. It retains completion time for every request.

- Batch membership can change after every model step.
- The same token capacity can produce different tail latency under different priorities.
- Fairness must be encoded as a scheduling rule and measured per request.


## 2. Derive the mechanism

At each Decode tick, continuous batching chooses up to `C` active sequences, advances each by one token, releases completed sequences, and admits more work. Shortest-remaining-token priority lowers average latency but can postpone long jobs. Adding an age term or service class changes the objective from pure token throughput to a declared fairness policy.

### Mechanism at a glance

```mermaid
flowchart TD
  A["request arrivals"] --> Q["waiting queue"]
  Q --> P["priority + admission"]
  P --> B["active token batch"]
  B --> G["one model step"]
  G --> C{"request complete?"}
  C -->|"no"| P
  C -->|"yes"| O["release slot and KV blocks"]
```

### Walk it step by step

1. **Observe arrivals.** Record when each request becomes eligible.
2. **Apply a named priority.** Select active work using remaining tokens, age, or class.
3. **Advance one iteration.** Generate one token for each scheduled sequence.
4. **Measure individuals.** Retain completion and waiting time instead of only aggregate tokens.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 4
LESSON_TITLE = 'Continuous Batching, Throughput, and Fairness'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260816
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | static groups that drain before admitting new work |
| Candidate | continuous shortest-remaining and age-aware scheduling |
| Held constant | arrival trace, per-request tokens, tick capacity, and tie-breaking |
| Measurements | makespan, mean/p95 latency, maximum wait, and completion order |
| Evidence | `numerical-model` |

**Experiment:** Replay one arrival trace through three schedulers and compare makespan, p95 completion, and maximum wait.


## 5. Inspect the experiment code

The simulation operates on explicit token quanta and keeps a per-request timeline. Its purpose is to make policy consequences visible; it does not substitute for native scheduler profiling.

Do not execute until the code matches the frozen table.


In [2]:
specs = [{"id":"A","arrival":0,"tokens":14},{"id":"B","arrival":0,"tokens":3},
         {"id":"C","arrival":1,"tokens":2},{"id":"D","arrival":2,"tokens":9},
         {"id":"E","arrival":3,"tokens":1},{"id":"F","arrival":5,"tokens":5}]; capacity = 3
def simulate(policy):
    remaining={x["id"]:x["tokens"] for x in specs}; arrival={x["id"]:x["arrival"] for x in specs}
    first={}; finish={}; active=[]; group=[]; t=0
    while remaining:
        available=[rid for rid in remaining if arrival[rid]<=t and rid not in active]
        if policy=="static":
            if not group: group=sorted(available,key=lambda x:(arrival[x],x))[:capacity]
            active=list(group)
        else:
            pool=list(set(active+available)); key=(lambda x:(remaining[x],arrival[x],x)) if policy=="shortest" else (lambda x:(remaining[x]-.8*(t-arrival[x]),arrival[x],x))
            active=sorted(pool,key=key)[:capacity]
        if not active: t+=1; continue
        for rid in list(active):
            first.setdefault(rid,t); remaining[rid]-=1
            if remaining[rid]==0:
                finish[rid]=t+1; del remaining[rid]; active.remove(rid)
                if rid in group: group.remove(rid)
        t+=1
    lat=[finish[x["id"]]-x["arrival"] for x in specs]; waits=[first[x["id"]]-x["arrival"] for x in specs]
    return {"makespan":max(finish.values()),"mean_latency":statistics.mean(lat),
            "p95_latency":percentile(lat,.95),"max_wait":max(waits),"finish":finish}
metrics={name:simulate(name) for name in ("static","shortest","age_aware")}
analysis=(f"Static/shortest/age-aware makespan was {metrics['static']['makespan']}/"
          f"{metrics['shortest']['makespan']}/{metrics['age_aware']['makespan']} ticks. Priority changed "
          "per-request wait even with identical capacity; tick duration is modeled.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Static makespan | 28.000000 |
| Continuous makespan | 15.000000 |
| Age-aware makespan | 15.000000 |
| Static p95 | 22.500000 |
| Shortest max wait | 0.000000 |
| Age-aware max wait | 0.000000 |


## 7. Explain the result

Static/shortest/age-aware makespan was 28/15/15 ticks. Priority changed per-request wait even with identical capacity; tick duration is modeled.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. A transparent allocator, scheduler, gateway, or policy model executed. It establishes the stated invariant, not native vLLM performance.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 4, "title": 'Continuous Batching, Throughput, and Fairness', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Continuous batching can reclaim finished slots immediately, while the priority rule—not batching alone—determines fairness.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 4,
  "title": "Continuous Batching, Throughput, and Fairness",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260816
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "static": {
      "makespan": 28,
      "mean_latency": 14.666666666666666,
      "p95_latency": 22.5,
      "max_wait": 18,
      "finish": {
        "B": 3,
        "A": 14,
        "E": 15,
        "C": 16,
        "D": 23,
        "F": 28
      }
    },
    "shortest": {
      "makespan": 15,
      "mean_latency": 5.833333333333333,
      "p95_latency": 13.5,
      "max_wait": 0,
      "finish": {
        "B": 3,
        "C": 3,
        "E": 4,
        "F": 10,
        "D": 11,
        "A": 15
      }
    },
    "age_aware": {
      "makespan": 15,
      "mean_latency": 5.833333333333333,
  

## 9. Make the bounded decision

> Continuous batching can reclaim finished slots immediately, while the priority rule—not batching alone—determines fairness.

**Acceptance/rollback:** Choose the policy whose tail and starvation metrics meet service-class gates at an acceptable throughput cost.

**Failure analysis:** Real Prefill steps have unequal cost, CUDA batches are not constant-duration ticks, and memory pressure can block admission. The trace is illustrative rather than a capacity prediction.


## 10. Extend the evidence

Replay production arrival timestamps with prompt lengths, streaming duration, priority class, cancellations, and measured per-step costs from the real engine.

The full boundary and references are in [`README.md`](README.md).
